# Notebook 25 — Evaluación Modelo C sobre dataset GAR

**TFM — Sistema de Detección de Amenazas Armadas en Vídeo**  
Oliver Legarreta García · Universitat Oberta de Catalunya

---

## Objetivo

Evaluar **Modelo C** (`yolov8m-seg`) sobre el dataset GAR con el mismo protocolo que Modelo B, para obtener la comparativa definitiva entre detección por bounding box y segmentación de instancias.

## Comparativa esperada

| Modelo | Arquitectura | Task | mAP@50 (val) | F1 GAR | FP | FN |
|--------|-------------|------|-------------|--------|-----|-----|
| **B** | yolov8m | Detección | 0.7789 | 0.7949 | 48 | 16 |
| **C** | yolov8m-seg | Segmentación | **0.9556** | ? | ? | ? |

**Hipótesis:** Modelo C debería reducir FP en N6–N9 (teléfono) y N10–N11 (botella) gracias al aprendizaje de la forma exacta del arma mediante segmentación.

**Dataset:** GAR test set — 140 clips positivos + 118 clips negativos  
**Modelo C:** `yolov8m_seg_C` · CONF=0.25 · Umbral clip=5 frames

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install ultralytics opencv-python
print('✅ Dependencias instaladas')

In [ ]:
import os
import json
import shutil
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
from ultralytics import YOLO

# ── CONFIG ────────────────────────────────────────────────────────────────────
POS_LIST   = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_test.txt'
NEG_LIST   = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/no_gun_test.txt'
OUT_DIR    = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/evaluation_modelo_c'

# Pesos Modelo C
MODEL_C_WEIGHTS = '/content/drive/MyDrive/TFM/experiments/weapon_seg/yolov8m_seg_C/weights/best.pt'

IMG_SIZE            = 640
CONF_WEAPON         = 0.25
IOU_NMS             = 0.7
DETECTION_THRESHOLD = 5
MAX_SECONDS         = 15

CAT_LABELS = {
    'N1': 'Walking empty hands',  'N2': 'Jogging',
    'N3': 'Running',              'N4': 'Sneaking empty hands',
    'N5': 'Phone relaxed',        'N6': 'Phone looking',
    'N7': 'Phone both hands',     'N8': 'Phone recording 1h',
    'N9': 'Phone recording 2h',   'N10': 'Water bottle relaxed',
    'N11': 'Drinking',            'N12': 'Holding heavy object',
}

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# Copiar pesos a /content/ para evitar desconexiones de Drive
!cp '{MODEL_C_WEIGHTS}' /content/modelo_c_best.pt

# Cargar Modelo C
weapon_model = YOLO('/content/modelo_c_best.pt')

print('✅ Config cargada')
print(f'   Modelo C: yolov8m-seg')
print(f'   CONF:     {CONF_WEAPON}')
print(f'   Umbral:   {DETECTION_THRESHOLD} frames')

---
## 1. Funciones auxiliares

In [ ]:
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0: return 0.0
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (aA + aB - inter)


def load_gt_boxes(label_json_path):
    with open(label_json_path) as f:
        data = json.load(f)
    gt = defaultdict(list)
    for ann in data['annotations']:
        x, y, w, h = ann['bbox']
        gt[ann['image_id']].append((x, y, x+w, y+h))
    return gt


def run_detector_on_video(video_path):
    """
    Ejecuta Modelo C frame a frame.
    Devuelve lista de (frame_idx, preds) donde preds = [(x1,y1,x2,y2,conf),...]
    Nota: yolov8m-seg devuelve tanto boxes como masks — usamos solo boxes para
    mantener el mismo protocolo de evaluación que Modelo B.
    """
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_frames = n_frames
    if MAX_SECONDS and fps > 0:
        max_frames = min(max_frames, int(MAX_SECONDS * fps))

    results = []
    frame_i = 0
    while True:
        ok, frame = cap.read()
        if not ok or frame_i >= max_frames:
            break

        r = weapon_model.predict(
            frame, imgsz=IMG_SIZE, conf=CONF_WEAPON,
            iou=IOU_NMS, verbose=False, device='cuda'
        )[0]

        preds = []
        if r.boxes is not None and len(r.boxes) > 0:
            for b in r.boxes:
                x1, y1, x2, y2 = map(float, b.xyxy[0])
                conf = float(b.conf[0])
                preds.append((x1, y1, x2, y2, conf))
        results.append((frame_i + 1, preds))
        frame_i += 1

    cap.release()
    return results, max_frames


print('✅ Funciones auxiliares definidas')

---
## 2. BLOQUE A — mAP a nivel de frame

In [ ]:
IOU_THRESHOLDS = np.arange(0.5, 1.0, 0.05)
all_tp = defaultdict(list)
all_fp = defaultdict(list)
total_gt = 0

print('\n' + '='*60)
print('BLOQUE A — mAP a nivel de frame (Modelo C)')
print('='*60)

pos_paths = [Path(l.strip()) for l in Path(POS_LIST).read_text().splitlines() if l.strip()]

for vp in pos_paths:
    if not vp.exists(): continue
    label_path = vp.parent / 'label.json'
    if not label_path.exists(): continue

    clip_id  = vp.parent.name
    local_in = f'/content/{clip_id}_eval.mp4'
    shutil.copy2(str(vp), local_in)

    gt_boxes = load_gt_boxes(label_path)
    total_gt += sum(len(v) for v in gt_boxes.values())

    frame_results, _ = run_detector_on_video(local_in)

    for (frame_idx, preds) in frame_results:
        gts = gt_boxes.get(frame_idx, [])
        for iou_thr in IOU_THRESHOLDS:
            matched_gt = set()
            for (px1, py1, px2, py2, conf) in sorted(preds, key=lambda x: -x[4]):
                best_iou, best_j = 0, -1
                for j, gt in enumerate(gts):
                    if j in matched_gt: continue
                    s = iou((px1, py1, px2, py2), gt)
                    if s > best_iou:
                        best_iou, best_j = s, j
                if best_iou >= iou_thr and best_j >= 0:
                    all_tp[iou_thr].append(1); all_fp[iou_thr].append(0)
                    matched_gt.add(best_j)
                else:
                    all_tp[iou_thr].append(0); all_fp[iou_thr].append(1)

    os.remove(local_in)
    print(f'  ✅ {clip_id}')

aps = {}
for iou_thr in IOU_THRESHOLDS:
    tp = sum(all_tp[iou_thr])
    fp = sum(all_fp[iou_thr])
    fn = total_gt - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    aps[iou_thr] = {'precision': precision, 'recall': recall}

map50   = aps[0.5]['precision']
map5095 = np.mean([v['precision'] for v in aps.values()])

print(f'\n  Total GT boxes  : {total_gt}')
print(f'  mAP@50          : {map50:.4f}')
print(f'  mAP@50:95       : {map5095:.4f}')
print(f'  Precision@50    : {aps[0.5]["precision"]:.4f}')
print(f'  Recall@50       : {aps[0.5]["recall"]:.4f}')

---
## 3. BLOQUE B — Clasificación a nivel de clip

In [ ]:
print('\n' + '='*60)
print(f'BLOQUE B — Clasificación clip-level (umbral={DETECTION_THRESHOLD} frames)')
print('='*60)

y_true, y_pred = [], []
clip_details   = []

def classify_clip(video_path):
    local_in = '/content/tmp_cls.mp4'
    shutil.copy2(str(video_path), local_in)
    frame_results, n_total = run_detector_on_video(local_in)
    os.remove(local_in)
    frames_with_gun = sum(1 for (_, preds) in frame_results if len(preds) > 0)
    pred = 1 if frames_with_gun >= DETECTION_THRESHOLD else 0
    return pred, frames_with_gun, n_total

# Positivos
print('\n  Procesando positivos...')
for vp in pos_paths:
    if not vp.exists(): continue
    clip_id = vp.parent.name
    pred, n_det, n_total = classify_clip(vp)
    y_true.append(1); y_pred.append(pred)
    clip_details.append({'clip': clip_id, 'true': 1, 'pred': pred,
                         'det_frames': n_det, 'total_frames': n_total,
                         'category': 'POS'})
    print(f'    {clip_id}: {n_det}/{n_total} frames → {"✅TP" if pred==1 else "❌FN"}')

# Negativos
neg_paths = [Path(l.strip()) for l in Path(NEG_LIST).read_text().splitlines() if l.strip()]
print(f'\n  Procesando negativos ({len(neg_paths)} clips)...')
for vp in neg_paths:
    if not vp.exists(): continue
    clip_id  = vp.parent.name
    category = clip_id.split('_')[0]
    pred, n_det, n_total = classify_clip(vp)
    y_true.append(0); y_pred.append(pred)
    clip_details.append({'clip': clip_id, 'true': 0, 'pred': pred,
                         'det_frames': n_det, 'total_frames': n_total,
                         'category': category})
    print(f'    {clip_id}: {n_det}/{n_total} frames → {"❌FP" if pred==1 else "✅TN"}')

# Métricas globales
y_true = np.array(y_true)
y_pred = np.array(y_pred)

TP = int(((y_true==1) & (y_pred==1)).sum())
TN = int(((y_true==0) & (y_pred==0)).sum())
FP = int(((y_true==0) & (y_pred==1)).sum())
FN = int(((y_true==1) & (y_pred==0)).sum())

accuracy  = (TP + TN) / len(y_true)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'\n  --- Métricas globales (clip-level) ---')
print(f'  Clips evaluados : {len(y_true)} ({TP+FN} pos / {TN+FP} neg)')
print(f'  Accuracy        : {accuracy:.4f}')
print(f'  Precision       : {precision:.4f}')
print(f'  Recall          : {recall:.4f}')
print(f'  F1              : {f1:.4f}')
print(f'\n  Matriz de confusión:')
print(f'                Pred POS   Pred NEG')
print(f'  Real POS    |   {TP:4d}   |   {FN:4d}  |')
print(f'  Real NEG    |   {FP:4d}   |   {TN:4d}  |')

---
## 4. BLOQUE C — FP por categoría negativa

In [ ]:
print('\n' + '='*60)
print('BLOQUE C — Falsos positivos por categoría (Modelo C)')
print('='*60)

neg_details = [d for d in clip_details if d['true'] == 0]
cat_stats   = defaultdict(lambda: {'total': 0, 'fp': 0})
for d in neg_details:
    cat = d['category']
    cat_stats[cat]['total'] += 1
    if d['pred'] == 1:
        cat_stats[cat]['fp'] += 1

# Comparativa con Modelo B
modelo_b_fp = {
    'N1':33.3,'N2':33.3,'N3':0.0,'N4':28.6,'N5':20.0,
    'N6':50.0,'N7':57.1,'N8':60.0,'N9':77.8,'N10':22.2,
    'N11':44.4,'N12':40.0
}

print(f"\n  {'Cat':<5} {'Descripción':<22} {'FP%_B':>7} {'FP%_C':>7} {'Δ':>7}")
print('  ' + '-'*55)
for cat in sorted(cat_stats, key=lambda x: int(x[1:])):
    s = cat_stats[cat]
    fp_c = s['fp'] / s['total'] * 100 if s['total'] > 0 else 0
    fp_b = modelo_b_fp.get(cat, 0)
    delta = fp_c - fp_b
    arrow = '↓' if delta < -5 else ('↑' if delta > 5 else '=')
    desc  = CAT_LABELS.get(cat, '')
    print(f"  {cat:<5} {desc:<22} {fp_b:>6.1f}% {fp_c:>6.1f}% {delta:>+6.1f}% {arrow}")

---
## 5. Guardar resultados

In [ ]:
import csv

# CSV de clips
csv_path = Path(OUT_DIR) / 'clip_results_modelo_c.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['clip','true','pred','det_frames','total_frames','category'])
    writer.writeheader()
    writer.writerows(clip_details)

# TXT de métricas
txt_path = Path(OUT_DIR) / 'results_modelo_c.txt'
with open(txt_path, 'w') as f:
    f.write('=== EVALUACIÓN MODELO C (yolov8m-seg) ===\n\n')
    f.write(f'CONF_WEAPON={CONF_WEAPON} | THRESHOLD={DETECTION_THRESHOLD}\n\n')
    f.write(f'--- BLOQUE A ---\n')
    f.write(f'mAP@50    : {map50:.4f}\n')
    f.write(f'mAP@50:95 : {map5095:.4f}\n\n')
    f.write(f'--- BLOQUE B ---\n')
    f.write(f'Accuracy  : {accuracy:.4f}\n')
    f.write(f'Precision : {precision:.4f}\n')
    f.write(f'Recall    : {recall:.4f}\n')
    f.write(f'F1        : {f1:.4f}\n')
    f.write(f'TP={TP} TN={TN} FP={FP} FN={FN}\n\n')
    f.write('--- BLOQUE C ---\n')
    for cat in sorted(cat_stats, key=lambda x: int(x[1:])):
        s = cat_stats[cat]
        fp_rate = s['fp'] / s['total'] * 100 if s['total'] > 0 else 0
        f.write(f'  {cat}: {s["fp"]}/{s["total"]} ({fp_rate:.1f}%)\n')

print(f'✅ Resultados guardados en: {OUT_DIR}')

# Tabla comparativa final
print('\n' + '='*65)
print('TABLA COMPARATIVA FINAL — Modelo B vs Modelo C')
print('='*65)
print(f"  {'Config':<30} {'F1':>7} {'Prec':>7} {'Rec':>7} {'FP':>5} {'FN':>5}")
print('  ' + '-'*60)
print(f"  {'B — yolov8m (detección)':<30} {'0.7949':>7} {'0.7209':>7} {'0.8857':>7} {'48':>5} {'16':>5}")
print(f"  {'C — yolov8m-seg (segm.)':<30} {f1:>7.4f} {precision:>7.4f} {recall:>7.4f} {FP:>5} {FN:>5}")
print()

if f1 > 0.7949:
    print(f'  ✅ Modelo C MEJORA el F1 de Modelo B (+{(f1-0.7949)*100:.2f} pp)')
elif f1 > 0.7800:
    print(f'  ≈  Modelo C comparable a Modelo B (diferencia < 1.5 pp)')
else:
    print(f'  ❌ Modelo C no supera a Modelo B — documentar como resultado')

print('\n✅ Evaluación completa.')